# Scrapers for MyDramaList

You will follow the instructions in Part 4 of Week 2 Tasks. In the top part of the notebook summarize through a table of content what you decided to do and then explain why.

**Author**: Crystal Zhao    
**Date**: 9/16

I decided to challenge my skills by first scraping an aggregated page, then scraping multiple aggregated pages (pagination). My research question is focused on the top rated movies (5000 results): What is the national distribution of top rated movies on this website? For each page, I would: 1) save the name, 2) rank, and 3) nationality of each movie. For "nationality", I would split up the "text-muted" element String and only extract the first word (`<span class="text-muted">Korean Movie - 2017</span>`).      

Then I scraped a dedicated page and built a dictionary, and this framework should be replicable across different dedicated pages. Finally, I scraped by scrolling and clicking on buttons.

**IMPORTANT: I chatted with Han about the situation with my computer but it seems to have low RAM and constantly runs into HTTP Reading Timeout errors when I did try to scrape everyhing from the top rated movies, for instance. Therefore, my approach in this notebook is to really tackle the fundamentals of scraping by breaking each process down into bits, and practicing my scrapers on fixed / smaller portions of the total HTML code. I combine Selenium and BeautifulSoup to get good practice with both!*** 

**Table of Contents**

1. [Scraping Aggregated Pages](#sec1)
2. [Scraping Across Pages](#sec2)
3. [Scraping Dedicated Pages](#sec3)
4. [Scraping by Scrolling](#sec4)

<a id="sec1"></a>

## Scraping Aggregated Pages

### a. Identifying the Movie Cards

Each movie card is a .box element,   
Each .box contains a .row element, with:  

1. ".ranking.pull-right"
2. ".text-primary-title" class with a hyperlink that directly leads to a dedicated page through a href (we should also save this so we can navigate to dedicated pages when necessary)
3. followed by the actual title
3. and finally ".text-muted" (e.g., Chinese Movie - 2019).

In [37]:
from seleniumbase import Driver
from selenium.webdriver.common.by import By
import re
import requests

url = "https://mydramalist.com/movies/top?page=2"

In [38]:
def fetch_page_content(url):
    """Fetches HTML content from a URL and checks status code."""
    response = requests.get(url) # call the function get with the URL, response is a Python object with a lot of attribues

    print(f"URL: {url}")
    print(f"Status Code: {response.status_code}")

    if response.status_code == 200: # check the status code to make sure we got the page from the server 
        return response.text # text is an attribute 
    else:
        print(f"Failed to fetch page. Status code: {response.status_code}")
        return None

In [39]:
page = fetch_page_content(url)

URL: https://mydramalist.com/movies/top?page=2
Status Code: 200


Now that we have the static page html, we can try to parse it with Selenium first to make sure we are looking in the right places for the information we want.

In [40]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(page, "html.parser")
top_movies = soup.find('div', class_='col-lg-8 col-md-8') # have to first narrow down to this so we ar enot including other boxes 
movies = top_movies.find_all("div", class_="box")
print(len(movies))

20


The tricky part for me right now is getting both the title and the link, so I am gonna investigate.

In [41]:
movie_1 = movies[0]
tags = movie_1.find_all('a')
print(len(tags)) 
print(tags)
tags[0]

3
[<a class="block" href="/24153-be-with-you">
<img alt="Be with You" class="img-responsive cover lazy" data-src="https://i.mydramalist.com/pnvlns.jpg?v=1"/>
</a>, <a href="/24153-be-with-you">Be with You</a>, <a class="btn simple btn-manage-list" data-id="24153" data-stats="mylist:24153" rel="nofollow"><span><i class="far fa-plus"></i></span></a>]


<a class="block" href="/24153-be-with-you">
<img alt="Be with You" class="img-responsive cover lazy" data-src="https://i.mydramalist.com/pnvlns.jpg?v=1"/>
</a>

What we need is the first a tag that is a block with the link, so we only need to do _find but let's verify that:

In [32]:
title_el = movie_1.find('a')
print(title_el)

<a class="block" href="/24153-be-with-you">
<img alt="Be with You" class="img-responsive cover lazy" data-src="https://i.mydramalist.com/pnvlns.jpg?v=1"/>
</a>


In [33]:
title_el.text

'\n\n'

We can't just call text, so we need to get the alternative description under image.

In [36]:
img_tag = movie_1.find('img')
img_tag['alt']

'Be with You'

In [ ]:
movie_data = []
for m in movies:
        movie_dict = {} 
        
        movie_dict['rank'] = m.find(class_='ranking pull-right').get_text()

        title_tag = movie_1.find('img')
        movie_dict['title'] = title_tag['alt']

        link_el = m.find('a')
        movie_dict['link'] = link_el['href']
        
        if movie_dict:
            movie_data.append(movie_dict)

In [38]:
movie_data[0]

{'rank': '#21', 'title': 'Be with You', 'link': '/24153-be-with-you'}

Now, to make sure we are also getting the country from text-muted.

In [26]:
text = 'Korean Movie - 2018'
text.split(' ')[0].strip()

'Korean'

Success! Now we should be able to wrap this in a function but in Selenium format.

### b. Defining the 'parse_movies' function

We would call this function after we get to each page, so we would run this function for each page of information that we get.

With this in mind, let's try to scrape just the first page (all movies) and save the information in 1 list.

In [42]:
from seleniumbase import Driver

def parse_movie(boxes):
    """
    Given a bunch of boxes on one page, extracts information from the entire movie through each box .
    """
    if not boxes:
        return []

    page_movies_data = []
    
    for movie in boxes:
        movie_dict = {}

        movie_dict['rank'] = movie.find_element(By.CSS_SELECTOR, '.ranking.pull-right').text
        title_tag = movie.find_element(By.TAG_NAME, 'img')
        movie_dict['title'] = title_tag.get_attribute('alt')
        
        link_el = movie.find_element(By.TAG_NAME,'a')
        movie_dict['link'] = link_el.get_attribute('href')

        description = movie.find_element(By.CSS_SELECTOR, '.text-muted').text
        movie_dict['country'] = description.split(' ')[0].strip()
        
        if movie_dict:
            page_movies_data.append(movie_dict)

    return page_movies_data

Now the challenge is to figure out how Selenium can get the "box" elements I need.



### c. Getting the Boxes using Selenium

In [43]:
from seleniumbase import Driver
from selenium.webdriver.common.by import By

url = "https://mydramalist.com/movies/top?page=2"

with Driver(browser='chrome', page_load_strategy='eager') as driver:
    driver.get(url)
    parent = driver.find_element(By.CSS_SELECTOR, '.col-lg-8.col-md-8')
    print("hello")
    boxes = parent.find_elements(By.CSS_SELECTOR,'.box')
    print(len(boxes))

hello
20


This means we got all 20 movies on this page. Now, let's call parse_movie on these boxes for page 1 first.

In [44]:
with Driver(browser='chrome', page_load_strategy='eager') as driver:
    driver.get(url)
    parent = driver.find_element(By.CSS_SELECTOR, '.col-lg-8.col-md-8')
    boxes = parent.find_elements(By.CSS_SELECTOR,'.box')
    print(f"Scraping {len(boxes)} boxes")
    page_movie_data = parse_movie(boxes) 

Scraping 20 boxes


In [45]:
page_movie_data[:2]

[{'rank': '#21',
  'title': 'Be with You',
  'link': 'https://mydramalist.com/24153-be-with-you',
  'country': 'Korean'},
 {'rank': '#22',
  'title': 'Along with the Gods 2: The Last 49 Days',
  'link': 'https://mydramalist.com/28305-along-with-the-gods-the-last-49-days',
  'country': 'Korean'}]

<a id="sec2"></a>

## 2. Scraping Across Pages

In [46]:
import math

url = "https://mydramalist.com/movies/top"

with Driver(browser='firefox', page_load_strategy='eager') as driver:
    driver.open(url)

    # 1. Extract total results count (e.g., "5000 results" -> 105)
    total_text = driver.get_text(".m-b-sm.pull-right") # This will be a string value
    total_results = int(total_text.split()[0])
    print(total_results)

    # 2. Calculate total pages (20 items per page)
    total_pages = math.ceil(total_results / 20)
    print(f"Total results: {total_results} | Total pages: {total_pages}")

    all_data = []
    # 4. Loop through each page URL, but just 5 for now to make sure it works 
    for page in range(1, 6):
        driver.open(f"{url}?page={page}")
        driver.sleep(1.0)

        # 5. Extract items on the current page
        parent = driver.find_element(By.CSS_SELECTOR, '.col-lg-8.col-md-8')
        boxes = parent.find_elements(By.CSS_SELECTOR,'.box')
        print(f"Scraping {len(boxes)} boxes")
        page_movie_data = parse_movie(boxes) 

        if page_movie_data:
            all_data.append(page_movie_data)
       

5000
Total results: 5000 | Total pages: 250
Scraping 20 boxes
Scraping 20 boxes
Scraping 20 boxes
Scraping 20 boxes
Scraping 20 boxes


Since my computer has really low RAM, I'm only running it on 5 pages to show that I can scrape across pages.

In [49]:
all_data[-1]

[{'rank': '#81',
  'title': 'Us and Them',
  'link': 'https://mydramalist.com/28319-us-and-them',
  'country': 'Chinese'},
 {'rank': '#82',
  'title': 'Memories of Murder',
  'link': 'https://mydramalist.com/990-memories-of-murder',
  'country': 'Korean'},
 {'rank': '#83',
  'title': 'Exhuma',
  'link': 'https://mydramalist.com/703025-pamyo',
  'country': 'Korean'},
 {'rank': '#84',
  'title': 'Rebound',
  'link': 'https://mydramalist.com/34649-rebound',
  'country': 'Korean'},
 {'rank': '#85',
  'title': 'Shark: The Beginning',
  'link': 'https://mydramalist.com/71505-shark',
  'country': 'Korean'},
 {'rank': '#86',
  'title': 'Juror 8',
  'link': 'https://mydramalist.com/28968-jury',
  'country': 'Korean'},
 {'rank': '#87',
  'title': 'Brave Citizen',
  'link': 'https://mydramalist.com/684849-brave-citizen',
  'country': 'Korean'},
 {'rank': '#88',
  'title': 'Kung Fu Hustle',
  'link': 'https://mydramalist.com/346-kung-fu-hustle',
  'country': 'Hong'},
 {'rank': '#89',
  'title': 'L

<a id="sec3"></a>

## 3. Scraping Dedicated Pages

Scraping the top rated movie:

**If we only wanted to stay at the top**:
movie['title'] = soup.find('div', class='film-title) but split at "(" and only keep the first item in the split list 

movie['year'] = soup.find('film-subtitle text-sm') but split at "space" and only keep the last item 

The ones below all use class = 'hfs'
movie['rating'] = soup.find(id, show-detailsxx) --> <div class="hfs" itempropx="aggregateRating" Ratings: <b style="font-weight:bold;" --> itempropx="ratingValue">9.2</b>/10 from 28,712 users 

movie['#_watchers'] = we only want to keep the number portion of this code (<div class="hfs"># of Watchers: <b>104,422</b></div>), so split at : and strip() the second.

movie['#_reviewers] = we only want to keep the text-primary portion of this (<div class="hfs">Reviews: <a class="text-primary" href="/30499-the-youthful-you-who-was-so-beautiful/reviews">134 users</a></div>)

movie['synopsis'] = find class='show-synopsis', the first <p> tag

**Directly scraping details**:   
The below are all uder class = show-detailsxss:
li class - all p-a-0
movie['Native Title'].   
movie['Director].   
movie['Screenwriter'].   
movie['Genres'].   
movie['Tags'].   
movie['Coutry'].   
movie['release date'].   
movie['duration'].   
movie['score'].   
movie['ranked'].   
movie['popularity'].    
movie['content rating'].   

I will use BeautifulSoup to scrape one dedicated page.

In [5]:
from bs4 import BeautifulSoup

In [6]:
url = 'https://mydramalist.com/30499-the-youthful-you-who-was-so-beautiful'
page = fetch_page_content(url)
soup = BeautifulSoup(page, 'html.parser')

URL: https://mydramalist.com/30499-the-youthful-you-who-was-so-beautiful
Status Code: 200


In [10]:
details = soup.find(class_='show-detailsxss')
details

<div class="show-detailsxss"> <ul class="list m-a-0"> <li class="list-item p-a-0 m-b-sm related-content"> <b class="inline">Related Content</b> <div class="title"> <a class="text-primary" href="/763893-ru-ci-mei-li" title="Ru Ci Mei Li">Ru Ci Mei Li</a>  (Chinese adaptation)  </div> </li> <li class="list-item p-a-0"><b class="inline">Native Title:</b> <a href="/30499-the-youthful-you-who-was-so-beautiful" title="少年的你">少年的你</a></li> <li class="list-item p-a-0"><b class="inline">Also Known As:</b> <span class="mdl-aka-titles">  Geng Hao De Ming Tian ,  In His Youth ,  Shao Nian De Ni ,  Shao Nian De Ni, Ru Ci Mei Li ,  The Youthful You ,  The Youthful You Who Was So Beautiful ,  少年的你, 如此美丽 ,  更好的明天   </span> </li> <li class="list-item p-a-0"><b class="inline">Director:</b> <a class="text-primary" href="/people/30541-derek-tsang">Derek Tsang</a> </li> <li class="list-item p-a-0"><b class="inline">Screenwriter:</b> <a class="text-primary" href="/people/65671-lam-wing-sam">Lam WIng Sam</a>,

The li tag means list item, and must always stand within a parent list item.

In [ ]:
all_items = details.find_all(class_='list-item p-a-0')
print(len(all_items))


14


<li class="list-item p-a-0" style="display:none;"><b class="inline">Watchers:</b> 104,422</li>

This is too disorganized! I'm gonna get the parent element instead.

In [34]:
specifics = details.find(class_='list m-a-0') # NOT find_all, because that gives you a list instead of a tag
type(specifics)

bs4.element.Tag

In [35]:
temp = []
for item in specifics.find_all('li'): # iterating through each list item
    text = item.text.split(':')[-1].strip()
    print(text)
    temp.append(text)


Related Content  Ru Ci Mei Li  (Chinese adaptation)
少年的你
Geng Hao De Ming Tian ,  In His Youth ,  Shao Nian De Ni ,  Shao Nian De Ni, Ru Ci Mei Li ,  The Youthful You ,  The Youthful You Who Was So Beautiful ,  少年的你, 如此美丽 ,  更好的明天
Derek Tsang
Lam WIng Sam,  Li Yuan,  Xu Yi Meng
Psychological,  Youth,  Drama
School Bullying, Coming Of Age, Social Commentary, High School, Teenager Female Lead, Poor Male Lead, Orphan Male Lead, Poor Female Lead, Smart Female Lead, Unusual Friendship (Vote tags)


In [36]:
temp

['Related Content  Ru Ci Mei Li  (Chinese adaptation)',
 '少年的你',
 'Geng Hao De Ming Tian ,  In His Youth ,  Shao Nian De Ni ,  Shao Nian De Ni, Ru Ci Mei Li ,  The Youthful You ,  The Youthful You Who Was So Beautiful ,  少年的你, 如此美丽 ,  更好的明天',
 'Derek Tsang',
 'Lam WIng Sam,  Li Yuan,  Xu Yi Meng',
 'Psychological,  Youth,  Drama',
 'School Bullying, Coming Of Age, Social Commentary, High School, Teenager Female Lead, Poor Male Lead, Orphan Male Lead, Poor Female Lead, Smart Female Lead, Unusual Friendship (Vote tags)']

Good! The rest are in the second list called hidden-md-up

In [37]:
more_items = details.find(class_='list m-a-0 hidden-md-up')
more_items

<ul class="list m-a-0 hidden-md-up"> <li class="list-item p-a-0"><b class="inline">Country:</b> China <i class="flag flags-c2"></i></li> <li class="list-item p-a-0"><b class="inline">Type:</b> Movie</li> <li class="list-item p-a-0"><b class="inline">Release Date:</b> Oct 25, 2019</li> <li class="list-item p-a-0"><b class="inline">Duration:</b> 2 hr. 18 min.</li> <li class="list-item p-a-0"><b class="inline">Score:</b> 9.2 <span class="hft">(scored by <a href="/30499-the-youthful-you-who-was-so-beautiful/statistics">28,712 users</a>)</span></li> <li class="list-item p-a-0"><b class="inline">Ranked:</b> #25</li> <li class="list-item p-a-0"><b class="inline">Popularity:</b> #127</li> <li class="list-item p-a-0"><b class="inline">Content Rating:</b> 15+ - Teens 15 or older</li> <li class="list-item p-a-0" style="display:none;"><b class="inline">Watchers:</b> 104,422</li> <li class="list-item p-a-0" style="display:none;"><b class="inline">Favorites:</b> 0</li> </ul>

In [ ]:
temp = []
for item in more_items.find_all('li'): # iterating through each list item
    text = item.text.split(':')[-1].strip()
    print(text)
    temp.append(text)

China
Movie
Oct 25, 2019
2 hr. 18 min.
9.2 (scored by 28,712 users)
#25
#127
15+ - Teens 15 or older
104,422
0


The last item, 0 refers to "favorites" but it does not appear on the webpage itself, so I am thinking of just dropping it.

In [48]:
top_movie = {}

title_tag = soup.find(class_='film-title')
top_movie['title'] = title_tag.text

details = soup.find(class_='show-detailsxss')
specifics = details.find(class_='list m-a-0') 
for item in specifics.find_all('li'): # iterating through each list item
    key = item.text.split(':')[0].strip()
    value = item.text.split(':')[-1].strip()
    top_movie[f"{key}"] = value
    print(value)

specifics_2 = details.find(class_='list m-a-0 hidden-md-up')
for item in specifics_2.find_all('li'):
    key = item.text.split(':')[0].strip()
    value = item.text.split(':')[-1].strip()
    top_movie[f"{key}"] = value
    print(value)

Related Content  Ru Ci Mei Li  (Chinese adaptation)
少年的你
Geng Hao De Ming Tian ,  In His Youth ,  Shao Nian De Ni ,  Shao Nian De Ni, Ru Ci Mei Li ,  The Youthful You ,  The Youthful You Who Was So Beautiful ,  少年的你, 如此美丽 ,  更好的明天
Derek Tsang
Lam WIng Sam,  Li Yuan,  Xu Yi Meng
Psychological,  Youth,  Drama
School Bullying, Coming Of Age, Social Commentary, High School, Teenager Female Lead, Poor Male Lead, Orphan Male Lead, Poor Female Lead, Smart Female Lead, Unusual Friendship (Vote tags)
China
Movie
Oct 25, 2019
2 hr. 18 min.
9.2 (scored by 28,712 users)
#25
#127
15+ - Teens 15 or older
104,422
0


In [49]:
top_movie

{'title': 'Better Days (2019)',
 'Related Content  Ru Ci Mei Li  (Chinese adaptation)': 'Related Content  Ru Ci Mei Li  (Chinese adaptation)',
 'Native Title': '少年的你',
 'Also Known As': 'Geng Hao De Ming Tian ,  In His Youth ,  Shao Nian De Ni ,  Shao Nian De Ni, Ru Ci Mei Li ,  The Youthful You ,  The Youthful You Who Was So Beautiful ,  少年的你, 如此美丽 ,  更好的明天',
 'Director': 'Derek Tsang',
 'Screenwriter': 'Lam WIng Sam,  Li Yuan,  Xu Yi Meng',
 'Genres': 'Psychological,  Youth,  Drama',
 'Tags': 'School Bullying, Coming Of Age, Social Commentary, High School, Teenager Female Lead, Poor Male Lead, Orphan Male Lead, Poor Female Lead, Smart Female Lead, Unusual Friendship (Vote tags)',
 'Country': 'China',
 'Type': 'Movie',
 'Release Date': 'Oct 25, 2019',
 'Duration': '2 hr. 18 min.',
 'Score': '9.2 (scored by 28,712 users)',
 'Ranked': '#25',
 'Popularity': '#127',
 'Content Rating': '15+ - Teens 15 or older',
 'Watchers': '104,422',
 'Favorites': '0'}

<a id="sec4"></a>

## 4. Scraping By Scrolling & Clicking Button

Some observations before going in: the comments section is at the very bottom of the page dedicated to the show. We would want to obtain the number of comments and also every comment. It seems that we do not need to click "load more comments" but it has been changed to infinite scroll.

PARENT ELEMENT: CSS_SELECTOR: .box.comments-box.post-comments

NUMBER: CSS_SELECTOR: .box.header.b-b (`<h3>Comments <span>(7294)</span></h3>`)

COMMENTS: CSS_SELECTOR: .box-body.thread-post-form
/ (ul) .comment-top
/ (li) .post.comment (for each item in the list, capture) --> .post-message -->  `<p>` tag

LOAD MORE BUTTON: CSS_SELECTOR: .box-footer
/  `<button type="button" class="el-button btn btn-block btn-default el-button--default"><!----><!----><span><strong>Load more comments</strong></span></button>`

Here's a reminder of how to find the button, but it's interesting that I don't really see an ID so I do have to use the very long class name": 

Let's first only get the page without loading more comments:

In [50]:
url = "https://mydramalist.com/18452-goblin"

with Driver(browser='firefox', page_load_strategy = 'eager') as driver:
    driver.open(url)
    driver.sleep(2)
    no_load_html = driver.get_page_source()

len(no_load_html)

109946

In [51]:
soup = BeautifulSoup(no_load_html, 'html.parser')
meta_comments_tag = soup.find(class_='box-header b-b')

In [54]:
print(meta_comments_tag)

<div class="box-header b-b"><h3>Comments <!-- --></h3> <p class="m-t"><small>MyDramaList is a space for respectful and thoughtful discussion. Harassment, hate speech, personal attacks, and inappropriate language are not allowed and may result in content removal or account action. Please keep things kind and civil.</small></p> <!-- --></div>


Now, let's press the button once.

In [55]:
url = "https://mydramalist.com/18452-goblin"

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

with Driver(browser='firefox', page_load_strategy = 'eager') as driver:
    driver.open(url)
    driver.sleep(2)

    # First scroll to make sure the comments section / button is in sight 
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    print('scrolled')
    driver.sleep(1)
    """
    The following represent my trial and error with finding the button...

    buttons = driver.find_elements(By.TAG_NAME, "button")
    # button = wait.until(EC.element_to_be_clickable((By.TAG_NAME, "button")))

    target_text = "Load more comments"
    button = next((btn for btn in buttons if btn.text == target_text), None)

    driver.execute_script("arguments[0].scrollIntoView(true);", button)


    print('success')
    print(type(button))
    
   # SeleniumBase finds the text, scrolls into view, and clicks automatically

    button = driver.find_element(By.CLASS_NAME, 'el-button--default')
    driver.sleep(2)
    """

    wait = WebDriverWait(driver, 10)

    # Then wait until the element is present in DOM and clickable
    button = wait.until(
        EC.element_to_be_clickable((By.CLASS_NAME, "el-button--default")) # Nina helped with this (unique class name)
        )

    # To ensure that nothing is blocking us from clicking the button
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", button)
    driver.sleep(0.5)

    # 4. Attempt standard click
    button.click()
    load_once_html = driver.get_page_source()


scrolled


In [56]:
len(load_once_html)

371465

This confirms that there is in fact more html now.

In [57]:
soup = BeautifulSoup(load_once_html, 'html.parser')
meta_comments_tag = soup.find(class_='box-header b-b')
parent = soup.find(class_='box comments-box post-comments')

In [59]:
print(parent)

<div class="box comments-box post-comments" id="cmtsapp" pid="18452" ptype="title" style="min-height: 100px;" title="Guardian: The Lonely and Great God"><div class="box-header b-b"><h3>Comments <span>(7294)</span></h3> <p class="m-t"><small>MyDramaList is a space for respectful and thoughtful discussion. Harassment, hate speech, personal attacks, and inappropriate language are not allowed and may result in content removal or account action. Please keep things kind and civil.</small></p> <!-- --></div> <div class="box-body b-b clear light"><div><a class="text-primary quick-login" href="/signin">Login</a> or <a class="text-primary" href="/signup">Register</a> to post comments</div></div> <div class="box-body thread-post-form"><ul class="post-list" id="comment-top"><li class="post comment" id="post-27465808"><!-- --> <div class="post-content"><div class="avatar"><span class="user in-link"><img class="user-avatar" src="https://i.mydramalist.com/Bd3wwV_1t.jpg"/></span></div> <div class="pos

In [ ]:
comments_list = parent.find(id='comment-top')
len(comments_list)

<ul class="post-list" id="comment-top"><li class="post comment" id="post-27465808"><!-- --> <div class="post-content"><div class="avatar"><span class="user in-link"><img class="user-avatar" src="https://i.mydramalist.com/Bd3wwV_1t.jpg"/></span></div> <div class="post-body"><div class="post-header p-b-xs"><a class="user-display text-primary in-link" href="/profile/Kevo" target="_blank"><b>Eeevo</b></a> <span class="mdl-utag"><span class="mdl-btag"><span class="mdl-verified" title="Verified Account"></span> <!-- --></span> <!-- --> <!-- --> <!-- --></span> <span class="date">5 days ago</span> <!-- --></div> <div class="post-message"><p>I like the plot but I’ve always seen them as a niece and uncle. I can’t imagine them as lovers.</p></div> <div class="post-actions"><button class="btn-like"><i class="like-heart"></i> 3
                </button> <button class="action-reply">Reply</button> <!-- --> <div class="action-menu dropdown"><button class="menu-toggle" data-toggle="dropdown" href="#"

This tells us that there are 105 items in the comments_list. We need to make sure they are actually comments though.

In [ ]:
comments_data = []

for comment in comments_list.find_all("li", class_="post comment", recursive=False): # I don't want to reach into children because those are replies
    post_content = comment.find(class_="post-content")
    message = post_content.find(class_="post-message")
    
    if message is not None:
        comments_data.append(message.get_text(" ", strip=True))

print(len(comments_data))
comments_data[:10]

92


['I like the plot but I’ve always seen them as a niece and uncle. I can’t imagine them as lovers.',
 "Watching this drama was genuinely devastating, and even long after the screen goes black, that heavy sadness lingers because the show pulled off a cruel trick disguised as a romance. Out of all the Asian dramas I have watched...including Chinese, Japanese, and Korean series, this is actually the only one that has ever made me cry real tears and completely filled me with raw emotion. Ripping away a character as pure, innocent, and completely full of love as Ji Eun-tak felt incredibly unfair. She spent her entire childhood suffering under an abusive family, finally found safety and joy in Kim Shin's arms, only to be brutally killed in an accident just as her real happiness was beginning. Crying over her death is completely natural because she was harmless and lovable in every single way, making her loss feel like an unnecessary punishment for someone who had already suffered so much.What

In [78]:
def scrape_comments(html): 
    """
    Given the full HTML page, scrapes all comments.
    """
    soup = BeautifulSoup(html, 'html.parser')
    # First, find the total number of comments
    
    # Second, find where the comments are:
    parent = soup.find(class_='box comments-box post-comments')
    if parent:
        print('Found parent!')

    comments_list = parent.find(id='comment-top')
    
    comments_data = {}
    count = 1
    for comment in comments_list.find_all("li", class_="post comment", recursive=False): # I don't want to reach into children because those are replies
        post_content = comment.find(class_="post-content")
        message = post_content.find(class_="post-message")
    
        if message is not None:
            comments_data[f"{count}"] = message.get_text(' ', strip=True)
        count+=1

    return comments_data

In [76]:
from bs4 import BeautifulSoup

In [79]:
scrape_comments(load_once_html)

Found parent!


{'1': 'I like the plot but I’ve always seen them as a niece and uncle. I can’t imagine them as lovers.',
 '2': "Watching this drama was genuinely devastating, and even long after the screen goes black, that heavy sadness lingers because the show pulled off a cruel trick disguised as a romance. Out of all the Asian dramas I have watched...including Chinese, Japanese, and Korean series, this is actually the only one that has ever made me cry real tears and completely filled me with raw emotion. Ripping away a character as pure, innocent, and completely full of love as Ji Eun-tak felt incredibly unfair. She spent her entire childhood suffering under an abusive family, finally found safety and joy in Kim Shin's arms, only to be brutally killed in an accident just as her real happiness was beginning. Crying over her death is completely natural because she was harmless and lovable in every single way, making her loss feel like an unnecessary punishment for someone who had already suffered so

Now we have successfully scraped all the comments from the full html code that loaded comments once.